In [ ]:
# Clone Repo 
import os

REPO_URL = "https://github.com/annieyp/cv-crowdcrush-detection.git"
REPO_DIR = "cv-crowdcrush-detection"

if "COLAB_RELEASE_TAG" in os.environ:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    %cd {REPO_DIR}/notebooks

In [ ]:
# Data Prep
import os
import sys
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

sys.path.append(os.path.abspath("../src"))
from dataset import CustomImageDataset

#fix parameters later
sigma, k, beta = 15, 3, 0.3

#add paths
GT_DIR_TRAIN = ""
IMG_DIR_TRAIN = ""
GT_DIR_TEST = ""
IMG_DIR_TEST = ""

train_set = CustomImageDataset(gt_dir="train gt path", img_dir="train imgs path", sigma=sigma, k=k, beta=beta)
test_set = CustomImageDataset(gt_dir="test gt path", img_dir="test imgs path", sigma=sigma, k=k, beta=beta)

train_loader = DataLoader(train_set, batch_size=1, shuffle=False)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

In [ ]:
# Training/Testing
import torch.nn as nn
import train
import csrnet

model = csrnet.CSRNet()
loss_fn = nn.MSELoss()

#fix parameters later
learning_rate = 1e-6
epochs = 10

#density-map regression gradients are huge 
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.95, weight_decay=5e-4)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train.train(train_loader, model, loss_fn, optimizer)
    train.test(test_loader, model, loss_fn)
print("Done!")

In [ ]:
# Export
torch.save(model, 'model.pth')